In [1]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation


class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_heat_consumer_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_heat_consumer_id=HC_heat_consumer_id
        self.net=net
        self._build_tespy_Cycles()

    def _build_tespy_Cycles(self):
        def my_ude(ude):
            dt_DHN=ude.params['dt']
            return ude.conns[0].calc_T()+dt_DHN-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        self.nw_Cooling_net = Network()
        self.nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Cooling_net_compressor = Compressor("compresor")
        self.Cooling_net_condenser = Condenser("condensador")
        self.Cooling_net_valve = Valve("valvula_expansion")
        self.Cooling_net_evaporator = HeatExchanger("evaporador")
        self.Cooling_net_cc=CycleCloser('CycleCloser')
        self.Cooling_net_source_consumer=Source("Source_Consumer ")
        self.Cooling_net_sink_consumer=Sink("Sink_consumer")
        self.Cooling_net_source_reseau=Source("Source_Reseau")
        self.Cooling_net_sink_reseau=Sink("Sink_Reseau")
        self.Cooling_net_c0=Connection(self.Cooling_net_valve, 'out1', self.Cooling_net_cc, 'in1', label='0')
        self.Cooling_net_c1 = Connection(self.Cooling_net_cc, 'out1', self.Cooling_net_evaporator, 'in2', label='1')
        self.Cooling_net_c2 = Connection(self.Cooling_net_evaporator, 'out2', self.Cooling_net_compressor, 'in1', label='2')
        self.Cooling_net_c3 = Connection(self.Cooling_net_compressor, 'out1', self.Cooling_net_condenser, 'in1', label='3')
        self.Cooling_net_c4 = Connection(self.Cooling_net_condenser, 'out1', self.Cooling_net_valve, 'in1', label='4')
        self.Cooling_net_c5=Connection(self.Cooling_net_condenser, 'out2',self.Cooling_net_sink_consumer, 'in1', label='5')
        self.Cooling_net_c6=Connection(self.Cooling_net_source_consumer, 'out1',self.Cooling_net_condenser , 'in2', label='6')
        self.Cooling_net_c7=Connection(self.Cooling_net_source_reseau, 'out1',self.Cooling_net_evaporator , 'in1', label='7')
        self.Cooling_net_c8=Connection(self.Cooling_net_evaporator, 'out1',self.Cooling_net_sink_reseau , 'in1', label='8')
        self.nw_Cooling_net.add_conns(self.Cooling_net_c0,  self.Cooling_net_c1,  self.Cooling_net_c2,  self.Cooling_net_c3,  self.Cooling_net_c4 , self.Cooling_net_c5,  self.Cooling_net_c6, self.Cooling_net_c7, self.Cooling_net_c8)
        self.Cooling_ude = UserDefinedEquation(
                    'my_ude', my_ude, my_ude_dependents, conns=[self.Cooling_net_c8, self.Cooling_net_c7],params={'dt': 5})
        self.nw_Cooling_net.add_ude(self.Cooling_ude)
        
        #Heating net 
        self.nw_Heating_net = Network()
        self.nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Heating_net_compressor = Compressor("compresor")
        self.Heating_net_condenser = Condenser("condensador")
        self.Heating_net_valve = Valve("valvula_expansion")
        self.Heating_net_evaporator = HeatExchanger("evaporador")
        self.Heating_net_cc=CycleCloser('CycleCloser')
        self.Heating_net_source_consumer=Source("Source_Consumer ")
        self.Heating_net_sink_consumer=Sink("Sink_consumer")
        self.Heating_net_source_reseau=Source("Source_Reseau")
        self.Heating_net_sink_reseau=Sink("Sink_Reseau")
        self.Heating_net_c0=Connection(self.Heating_net_valve, 'out1', self.Heating_net_cc, 'in1', label='0')
        self.Heating_net_c1 = Connection(self.Heating_net_cc, 'out1', self.Heating_net_evaporator, 'in2', label='1')
        self.Heating_net_c2 = Connection(self.Heating_net_evaporator, 'out2', self.Heating_net_compressor, 'in1', label='2')
        self.Heating_net_c3 = Connection(self.Heating_net_compressor, 'out1', self.Heating_net_condenser, 'in1', label='3')
        self.Heating_net_c4 = Connection(self.Heating_net_condenser, 'out1', self.Heating_net_valve, 'in1', label='4')
        self.Heating_net_c5 = Connection(self.Heating_net_evaporator, 'out1', self.Heating_net_sink_consumer, 'in1', label='5')
        self.Heating_net_c6 = Connection(self.Heating_net_source_consumer, 'out1', self.Heating_net_evaporator, 'in1', label='6')  
        self.Heating_net_c7 = Connection(self.Heating_net_source_reseau, 'out1', self.Heating_net_condenser, 'in2', label='7')
        self.Heating_net_c8 = Connection(self.Heating_net_condenser, 'out2', self.Heating_net_sink_reseau, 'in1', label='8')
        self.nw_Heating_net.add_conns(self.Heating_net_c0,  self.Heating_net_c1,  self.Heating_net_c2,  self.Heating_net_c3,  self.Heating_net_c4 , self.Heating_net_c5,  self.Heating_net_c6, self.Heating_net_c7, self.Heating_net_c8)
        self.Heating_ude = UserDefinedEquation(
                        'my_ude_4', my_ude, my_ude_dependents, conns=[self.Heating_net_c7, self.Heating_net_c8],params={'dt': 5})
        self.nw_Heating_net.add_ude(self.Heating_ude)
        
    def solve_cycle(self,mode,Q_consumer,eta_s,T_nework_in,T_cons,dt_DHN):
        #The pressure values for the consumer and district heating side in the heating pump are arbitrary values
        self.mode=mode
        T_cons_in = T_cons[0]
        T_cons_out = T_cons[1]
        T_DH_in = T_nework_in
        if self.mode=="COOLING_NET":
            self.Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5,Q=Q_consumer)
            self.Cooling_net_compressor.set_attr(eta_s=eta_s)
            self.Cooling_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Cooling_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Cooling_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Cooling_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Cooling_net._set_p_range([p_triple, p_high])
            self.nw_Cooling_net._set_h_range([h_min,h_max])
            self.Cooling_ude.params['dt'] = dt_DHN
            self.nw_Cooling_net.solve('design')        
        elif self.mode=="HEATING_NET":
            self.Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5,Q=Q_consumer)
            self.Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Heating_net_compressor.set_attr(eta_s=eta_s)
            self.Heating_net_c2.set_attr(fluid={self.refrigerant: 1},td_dew=5)
            # 6. Parámetros del Consumidor 
            self.Heating_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Heating_net_c6.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Heating_net_c7.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Heating_net._set_p_range([p_triple, p_high])
            self.nw_Heating_net._set_h_range([h_min,h_max])
            self.Heating_ude.params['dt'] = dt_DHN
            self.nw_Heating_net.solve('design')
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "qext_w"] =abs( self.Cooling_net_evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_heat_consumer_id,"deltat_k"]=dT_water
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "qext_w"] = self.Heating_net_condenser.Q.val
            self.net.heat_consumer.at[self.HC_heat_consumer_id,"deltat_k"]=dT_water

        else: 
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_heat_consumer_id, "in_service"] =False

    def get_main_results(self):
        if self.mode=="COOLING_NET":
            results = {
                "Q_consumer": abs(self.Cooling_net_condenser.Q.val),
                "Q_network": abs(self.Cooling_net_evaporator.Q.val),
                "W_compressor":self.Cooling_net_compressor.P.val ,
                "COP": abs(self.Cooling_net_condenser.Q.val)/self.Cooling_net_compressor.P.val 
            }
        elif self.mode=="HEATING_NET":
            results = {
                "Q_consumer": abs(self.Heating_net_evaporator.Q.val),
                "Q_network": abs(self.Heating_net_condenser.Q.val),
                "COP": abs(self.Heating_net_evaporator.Q.val)/self.Heating_net_compressor.P.val 
            }
        else:
            results = {}
        return results

    def get_results_solver(self):
        if self.mode=="COOLING_NET":
            self.nw_Cooling_net.print_results()
        elif self.mode=="HEATING_NET":
            self.nw_Heating_net.print_results()
        else:
            return None


In [2]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_Ida")
j_HP_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_Ida")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_ret")

#Plant-Nodo
u=0.35 / (np.pi * 0.15)
T_ext=25+273.15
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=6.8, inner_diameter_mm=150, u_w_per_m2k=u, text_k=T_ext, name="Tubo_Ida_Plant_Nodo_1",k_mm=0.1
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=6.8, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="Tubo_RetornoPlant_Nodo_1",k_mm=0.1)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=4.7, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="nodo_1_nodo_2",k_mm=0.1
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=4.7, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="nodo_2_ret_nodo_1_ret",k_mm=0.1
)

#Node HP
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_1,
    length_km=4.6, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="Tubo_Ida_nodo_2_cons_1_ida",k_mm=0.1
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_nodo_2_ret,
    length_km=4.6, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="Tubo_Retorno_nodo_2_cons_1_ret",k_mm=0.1
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_2,
    length_km=4.5, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="Tubo_Ida_nodo_2_cons_2_ida",k_mm=0.1
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=4.5, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=T_ext, name="Tubo_Retorno_nodo_2_cons_2_ret",k_mm=0.1
)

#Heat Pumps
#Heat pump: extraction
HP_1=pp.create_heat_consumer(
    net,
    from_junction=j_HP_1 ,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat pump: extraction
HP_2=pp.create_heat_consumer(
    net,
    from_junction=j_HP_2 ,
    to_junction=j_HP_ret_2,
    qext_w=150000,
    deltat_k=50,
    name="HP_2_EXT"
)
#Plant: 
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=310 ,name='Grid')

In [3]:
T_network_inital_guess_1=25
T_network_inital_guess_2=15


In [4]:
HP_1_object=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_heat_consumer_id=HP_1)
HP_2_object=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_heat_consumer_id=HP_2)

In [5]:
T_network_HP_1=T_network_inital_guess_1
T_network_HP_2=T_network_inital_guess_2
T_network_HP_1_loop=T_network_inital_guess_1
T_network_HP_2_loop=T_network_inital_guess_2
tolerance=1E-6
error_1=10
error_2=10
iterations=0
while error_1>tolerance or error_2>tolerance:
    HP_1_object.solve_cycle(mode="COOLING_NET",Q_consumer=-15000,eta_s=0.95,T_nework_in=T_network_HP_1,T_cons=[50,60],dt_DHN=5)
    HP_2_object.solve_cycle(mode="HEATING_NET",Q_consumer=-150000,eta_s=0.95,T_nework_in=T_network_HP_2,T_cons=[35,25],dt_DHN=10)
    print("iteration")
    HP_1_object.interaction_simulation(dT_water=5)
    HP_2_object.interaction_simulation(dT_water=-10)
    pp.pipeflow(net=net,mode='bidirectional')
    T_network_HP_1=float(net.res_heat_consumer.at[HP_1_object.HC_heat_consumer_id,"t_from_k"]-273.15)
    T_network_HP_2=float(net.res_heat_consumer.at[HP_2_object.HC_heat_consumer_id,"t_from_k"]-273.15)
    error_1=abs(T_network_HP_1_loop-T_network_HP_1)
    error_2=abs(T_network_HP_2_loop-T_network_HP_2)
    print(T_network_HP_1)
    print(T_network_HP_2)
    T_network_HP_1_loop=T_network_HP_1
    T_network_HP_2_loop=T_network_HP_2
    iterations+=1
print("total iterations"+ " were: "+ str(iterations))


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 9.73e+05   | 0 %        | 1.76e+00   | 1.81e+06   | 4.45e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 3.70e+04   | 15 %       | 9.80e-01   | 4.34e+05   | 1.13e+04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 5.88e+03   | 24 %       | 2.64e-01   | 6.20e+04   | 2.57e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 7.04e+01   | 46 %       | 2.36e-04   | 3.54e+03   | 1.34e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 1.83e-01   | 74 %       | 7.95e-07   | 9.82e+00   | 3.66e-01   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 6.54e-06   | 100 %      | 5.73e-12   | 7.50e-05   | 1.97e-06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 7     | 4.80e-10   | 100 %      | 1.42e-13   | 2.47e-09   | 5.25e-09   | 0.00e+00   | 0.00e+00   | 0.0

Found singularity in Jacobian matrix, calculation aborted! The setup of your problem seems to be solvable. It failed due to partial derivatives in the Jacobian being zero where a non-zero was expected, or vice versa. This usually lies in starting value selection or bad convergence.
  The following equation/variable pairs may have an unexpected zero/non-zero partial derivative:  condensador.energy_balance_constraints: m6
The following variables are not associated with any equation:
  m6



iteration
26.429466780994403
25.065009007039464

 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 3.40e+03   | 27 %       | 3.92e-03   | 2.24e+04   | 6.04e+03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 2.28e+01   | 51 %       | 1.08e-03   | 3.94e+02   | 1.41e+01   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 2.30e-03   | 95 %       | 4.36e-08   | 1.15e-01   | 4.10e-03   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 6.26e-06   | 100 %      | 2.53e-12   | 6.65e-07   | 5.55e-06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 3.20e-10   | 100 %      | 1.15e-13   | 6.75e-07   | 2.45e-08   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 5, Calculation time: 0.02 s, Iterations per second: 204.19

 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | en

In [6]:
HP_1_object.solve_cycle(mode="COOLING_NET",Q_consumer=-15000,eta_s=0.95,T_nework_in=T_network_HP_1,T_cons=[50,60],dt_DHN=5)
HP_2_object.solve_cycle(mode="HEATING_NET",Q_consumer=-150000,eta_s=0.95,T_nework_in=T_network_HP_2,T_cons=[35,25],dt_DHN=10)
print("iteration")
HP_1_object.interaction_simulation(dT_water=5)
HP_2_object.interaction_simulation(dT_water=-10)
pp.pipeflow(net=net,mode='bidirectional')


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 5.45e-05   | 100 %      | 7.73e-11   | 3.94e-04   | 9.38e-05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 5.74e-10   | 100 %      | 8.15e-13   | 5.04e-09   | 2.90e-08   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 6.19e-10   | 100 %      | 2.05e-12   | 2.47e-09   | 7.32e-08   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 3.56e-10   | 100 %      | 1.28e-11   | 1.00e-09   | 4.57e-07   | 0.00e+00   | 0.00e+00   | 0.00e+00   
Total iterations: 4, Calculation time: 0.02 s, Iterations per second: 193.30

 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 7.30e-05   | 100

In [7]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,12259.097887,NaN,5.0,NaN,True,heat_consumer
1,HP_2_EXT,4,9,-168599.710491,NaN,-10.0,NaN,True,heat_consumer


In [8]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.494996,3.004177,303.138310,298.138310,298.138310,0.586444,-0.586444,0.000589,5.0,12259.097887
1,2.348889,3.148857,306.912347,316.912347,316.912347,4.033366,-4.033366,0.004064,-10.0,-168599.710491


In [9]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,12259.097887,NaN,5.0,NaN,True,heat_consumer
1,HP_2_EXT,4,9,-168599.710491,NaN,-10.0,NaN,True,heat_consumer


In [10]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.494996,3.004177,303.138310,298.138310,298.138310,0.586444,-0.586444,0.000589,5.0,12259.097887
1,2.348889,3.148857,306.912347,316.912347,316.912347,4.033366,-4.033366,0.004064,-10.0,-168599.710491


In [11]:
net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.5,3.0,310.261715,310.0,310.0,4.61981,-4.61981,0.004651,0.261715,-5053.60804


In [12]:
net.res_junction

,p_bar,t_k
0,3.000000,310.000000
1,2.704265,308.625869
2,2.499679,307.770399
3,2.494996,303.138310
4,2.348889,306.912347
5,2.500000,310.261715
6,2.795328,311.850456
7,2.999226,313.068798
8,3.004177,298.138310
9,3.148857,316.912347


In [13]:
net.pipe

,name,from_junction,to_junction,std_type,length_km,inner_diameter_mm,outer_diameter_mm,k_mm,loss_coefficient,u_w_per_m2k,text_k,sections,in_service,type
0,Tubo_Ida_Plant_Nodo_1,0,1,None,6.8,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
1,Tubo_RetornoPlant_Nodo_1,6,5,None,6.8,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
2,nodo_1_nodo_2,1,2,None,4.7,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
3,nodo_2_ret_nodo_1_ret,7,6,None,4.7,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
4,Tubo_Ida_nodo_2_cons_1_ida,2,3,None,4.6,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
5,Tubo_Retorno_nodo_2_cons_1_ret,8,7,None,4.6,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
6,Tubo_Ida_nodo_2_cons_2_ida,2,4,None,4.5,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe
7,Tubo_Retorno_nodo_2_cons_2_ret,9,7,None,4.5,150.0,150.0,0.1,0.0,0.742723,298.15,1,True,pipe


In [14]:
net.res_pipe

,v_mean_m_per_s,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,reynolds,lambda,dp_friction_loss_bar
0,0.263124,3.000000,2.704265,310.000000,308.625869,308.625869,4.619810,-4.619810,0.004650,55828.295100,0.018967,0.295735
1,0.263291,2.795328,2.500000,311.850456,310.261715,310.261715,4.619810,-4.619810,0.004653,57748.702091,0.018929,0.295328
2,0.263018,2.704265,2.499679,308.625869,307.770399,307.770399,4.619810,-4.619810,0.004648,54665.700640,0.018992,0.204586
3,0.263426,2.999226,2.795328,313.068798,311.850456,311.850456,4.619810,-4.619810,0.004655,59393.716884,0.018898,0.203898
4,0.033358,2.499679,2.494996,307.770399,303.138310,303.138310,0.586444,-0.586444,0.000589,6552.337691,0.027588,0.004683
5,0.033285,3.004177,2.999226,298.138310,313.068798,298.143937,0.586444,-0.586444,0.000588,5609.501218,0.029230,0.004951
6,0.229563,2.499679,2.348889,307.770399,306.912347,306.912347,4.033366,-4.033366,0.004057,46881.375990,0.019186,0.150790
7,0.230318,3.148857,2.999226,316.912347,313.068798,315.239032,4.033366,-4.033366,0.004070,55404.142504,0.018976,0.149630


In [15]:
results_1=HP_1_object.get_main_results()
results_2=HP_2_object.get_main_results()

In [16]:
print(results_1)

{'Q_consumer': 15000.0, 'Q_network': 12259.097886955557, 'W_compressor': 2740.9021130443866, 'COP': 5.4726507483111595}


In [17]:
15000.0/1685.6799401256844

8.898486386971895

In [18]:
print(results_2)

{'Q_consumer': 150000.0, 'Q_network': 168599.71049107183, 'COP': 8.064641655148417}


In [19]:
HP_1_object.get_results_solver()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 2.74e+03 | 3.31e+00 | -1.32e+01 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  

In [20]:
HP_2_object.get_results_solver()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 1.86e+04 | 2.23e+00 | -7.06e+00 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  

In [21]:
pipe_ida_1

np.int64(0)

In [22]:
pipe_ida_2

np.int64(2)

In [44]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,12259.097887,NaN,5.0,NaN,True,heat_consumer
1,HP_2_EXT,4,9,-168599.710491,NaN,-10.0,NaN,True,heat_consumer


In [45]:
net.heat_consumer.loc[net.heat_consumer["name"]=="HP_1_EXT","deltat_k"]=7

In [46]:
net.heat_consumer

,name,from_junction,to_junction,qext_w,controlled_mdot_kg_per_s,deltat_k,treturn_k,in_service,type
0,HP_1_EXT,3,8,12259.097887,NaN,7.0,NaN,True,heat_consumer
1,HP_2_EXT,4,9,-168599.710491,NaN,-10.0,NaN,True,heat_consumer


In [47]:
pp.pipeflow(net,mode="bidirectional")

In [49]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.531511,2.967541,301.955095,294.955095,294.955095,0.418785,-0.418785,0.000420,7.0,12259.097887
1,2.383448,3.114185,306.843841,316.843841,316.843841,4.033371,-4.033371,0.004064,-10.0,-168599.710491
